# UdaPlay — Part 1: RAG Pipeline

This notebook builds the retrieval layer the agent will use in Part 2.

We will:
1. Load the local game catalog (`games/games.json`).
2. Embed each record with OpenAI `text-embedding-3-small`.
3. Persist the vectors in a ChromaDB store on disk.
4. Run a few semantic-search smoke tests.


## 1. Environment

Make sure your `.env` file contains:

```
OPENAI_API_KEY="sk-..."
TAVILY_API_KEY="tvly-..."
```


In [1]:
from dotenv import load_dotenv
import os

load_dotenv()  # reads .env in the project root
assert os.getenv("OPENAI_API_KEY") is not None, "OPENAI_API_KEY missing"
print("OpenAI key loaded:", os.getenv("OPENAI_API_KEY")[:7] + "...")

OpenAI key loaded: sk-proj...


## 2. Inspect the game catalog

In [2]:
import json
from pathlib import Path

with open("games/games.json") as f:
    games = json.load(f)

print(f"{len(games)} games loaded")
games[0]

18 games loaded


{'id': 'fifa-21',
 'name': 'FIFA 21',
 'platform': ['PlayStation 4',
  'Xbox One',
  'Nintendo Switch',
  'PC',
  'PlayStation 5',
  'Xbox Series X/S'],
 'genre': 'Sports',
 'publisher': 'Electronic Arts',
 'developer': 'EA Vancouver, EA Romania',
 'release_year': 2020,
 'description': 'FIFA 21 is a football simulation video game developed by EA Vancouver and EA Romania and published by Electronic Arts. It features improved player movement, refined attacking mechanics, and an Ultimate Team mode with co-op support.'}

In [3]:
%pip install -r requirements.txt

Defaulting to user installation because normal site-packages is not writeable
Note: you may need to restart the kernel to use updated packages.


## 3. Build the vector store

The `GameVectorStore` class wraps a Chroma `PersistentClient` and uses OpenAI as the embedding function.
It stores both the embedded document text and structured metadata for filtering.

In [4]:
from lib.vectorstore import GameVectorStore

store = GameVectorStore(persist_dir="chromadb_store", collection_name="games")
store.reset()  # start clean each run; comment out to keep prior embeddings
n = store.load_from_json("games/games.json")
print(f"Indexed {n} games. Collection size: {store.count()}")

Indexed 18 games. Collection size: 18


## 4. Semantic-search smoke tests

In [5]:
def show(results):
    for r in results:
        sim = f"{r['similarity']:.3f}" if r['similarity'] is not None else "n/a"
        print(f"[{sim}] {r['metadata']['name']} ({r['metadata']['release_year']}) — {r['metadata']['developer']}")

show(store.query("Who developed FIFA 21?", n_results=3))

[0.722] FIFA 21 (2020) — EA Vancouver, EA Romania
[0.330] Minecraft (2011) — Mojang Studios
[0.301] Cyberpunk 2077 (2020) — CD Projekt Red


In [6]:
show(store.query("When was God of War Ragnarok released?", n_results=3))

[0.728] God of War Ragnarok (2022) — Santa Monica Studio
[0.365] Hades (2020) — Supergiant Games
[0.361] Elden Ring (2022) — FromSoftware


In [7]:
show(store.query("What platform was Pokemon Red launched on?", n_results=3))

[0.689] Pokemon Red (1996) — Game Freak
[0.312] Red Dead Redemption 2 (2018) — Rockstar Games
[0.307] Super Mario Odyssey (2017) — Nintendo EPD


In [8]:
# Out-of-distribution: should return low-similarity matches
show(store.query("What is Rockstar Games working on right now?", n_results=3))

[0.516] Red Dead Redemption 2 (2018) — Rockstar Games
[0.515] Grand Theft Auto V (2013) — Rockstar North
[0.352] Cyberpunk 2077 (2020) — CD Projekt Red


## 5. What's next

In Part 2 we wrap this store in an agent that can also evaluate the quality of the retrieved
documents and fall back to a Tavily web search when the local catalog is insufficient.